# DashAnalysis Quickstart

Slicing, line cuts and volumes from HKL point-cloud data.

`load_data` · `show_meta` · `show_point_cloud` · `slice_data` · `show_slice` · `line_cut` · `create_vol` · `show_vol`

Set `filename` below to one of your scans.

## Setup

**Backend.** `%matplotlib widget` enables the interactive line cut (needs `ipympl`). PyVista renders 3D inline.

In [ ]:
%matplotlib widget

import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt
pv.set_jupyter_backend('html')


**Session.** `DashAnalysis` remembers the last slice, so `line_cut` can reuse it.

In [ ]:
from dashpva.utils import DashAnalysis

da = DashAnalysis()

## Load data and inspect metadata

**Load.** Returns a `Data` object with `.points` (N×3 HKL), `.intensities`, `.metadata`.

Set this to one of your own scans.

In [ ]:
# Set this to your HDF5 data file
filename = "your_data_file.h5"

data = da.load_data(filename)

### Metadata overview

You can access a metadata dict on the Data object and also request formatted metadata via `da.show_meta()`.

**Metadata.** `data.metadata` is the raw dict; `show_meta` formats the whole file.

In [ ]:
# Inspect a specific metadata key (example: grid)
try:
    print("Metadata keys:", list(data.metadata.keys())[:10])
    print("Grid metadata:", data.metadata.get('grid'))
except Exception as e:
    print("No metadata available on Data object:", e)

# Show formatted metadata (text or dict)
_ = da.show_meta(filename, style="text")
_ = da.show_meta(filename, style="dict")
# _ = da.show_meta(filename, style="json")  # if you prefer JSON-style output


## Point cloud

Notebook rendering caps at 5 million points for interactivity.

**Point cloud.** Renders measured points without gridding.

Keep `opacity_range=(0.0, 1.0)` — without it sparse clouds can render invisible.

In [ ]:
# Basic point cloud rendering (Data object)
# Note: pass opacity_range to avoid LUT range issues
da.show_point_cloud(
    data,
    cmap='viridis',
    point_size=1.5,
    opacity=0.15,
    opacity_range=(0.0, 1.0),
    render_points_as_spheres=False,
    show_bounds=True
)

# Advanced point cloud rendering with intensity range and spherical glyphs
# You can also pass (points, intensities) explicitly
da.show_point_cloud(
    (data.points, data.intensities),
    clim=(100.0, 50000.0),            # controls color scaling
    cmap='jet',
    point_size=2.0,
    opacity=1.0,
    opacity_range=(0.2, 0.8),         # intensity-based opacity range
    render_points_as_spheres=True,
    axes_labels=('H','K','L'),
    show_bounds=True
)

print(f"Points: {data.points.shape}, Intensities range: {float(np.min(data.intensities)):.2f} - {float(np.max(data.intensities)):.2f}")


## 2D slices

Demonstrate canonical planes, custom normals/origins, custom HKL axes, slab thickness, and intensity-based filtering.

**Slice — preset plane.** `hkl='HK' | 'HL' | 'KL'` sets the in-plane axes. `shape` is the raster resolution, not a data size.

In [ ]:
# 1) Canonical HL plane with grid and HKL labels
#    HL plane: U=H, V=L, normal aligned with K
sl_hl = da.slice_data(
    data=data,
    hkl='HL',                 # HL plane preset
    shape=(256, 256),         # raster resolution (rows, cols)
    show=True,                # display via show_slice
    axis_display='hkl',       # formatted HKL labels
    show_grid=True
)


**Slice — any plane.** Give `hkl` a point and `normal` a direction to cut off-axis.

In [ ]:
# 2) Custom plane: specify origin (H,K,L) and normal vector
sl_custom_plane = da.slice_data(
    data=data,
    hkl=(0.40, 0.25, 0.70),   # plane origin in HKL
    normal=(0.1, 0.9, 0.3),   # plane normal
    shape=(256, 256),
    show=True,
    axis_display='uv',        # display U/V labels
    show_grid=True
)


**Slice — explicit axes.** `axes=(u, v, normal)` in HKL. Labels are derived and stored on the slice.

In [ ]:
# 3) Custom HKL in-plane axes with automatic axis labeling
#    Provide (u_hkl, v_hkl) and optionally normal; labels are formatted and stored on the slice
sl_axes = da.slice_data(
    data=data,
    axes=((1, 1, 0), (0, 0, 1), (1, -1, 0)),  # u = H+K, v = L; normal provided
    shape=(300, 300),
    show=True,
    axis_display='hkl',
    show_grid=True
)
try:
    print("Stored slice_shape:", sl_axes.field_data.get('slice_shape'))
    print("U-axis label:", sl_axes.field_data.get('slice_u_label'))
    print("V-axis label:", sl_axes.field_data.get('slice_v_label'))
except Exception as e:
    print("Axis labels not available:", e)


**Slice — slab.** A zero-thickness plane catches almost nothing. `slab_thickness` widens it to ±t.

In [ ]:
# 4) Thick slab selection around a plane with intensity filtering
#    Use slab_thickness to include points within ±thickness of plane
sl_slab = da.slice_data(
    data=(data.points, data.intensities),
    hkl='HL',                  # HL plane
    shape=(512, 512),
    #slab_thickness=3.0,        # include points within ±2.0 of plane (remove)
    intensity_range=(11,15),  # pre-filter contributing points (remove)
    show=False                 # we'll render separately with custom display limits
)

da.show_slice(
    sl_slab,
    cmap='jet',
    clim=(0, 15),           # display limits
    min_intensity=None,
    max_intensity=None,
    axis_display='hkl',
    show_grid=True
)


## Returned slice images

You can request a raster image and physical extent from show_slice and reuse them directly for line cuts.

**Get the image.** `return_image=True` gives `(img, extent)` — feed that to `line_cut(vol=...)` to reuse the same pixels.

In [ ]:
img, extent = da.show_slice(sl_axes, return_image=True)
print("Returned image shape:", img.shape)
print("Extent [Umin, Umax, Vmin, Vmax]:", extent)


## Line cuts

Operate on the last slice, on a `(img, extent)` pair, or on a slice mesh.

**Line cut — preset.** `'zero'` with `param=(0.0, 'x')` holds V, traverses U. `width_px` averages neighbouring rows.

In [ ]:
# 1) Horizontal line cut at fixed V ("zero" preset)
lc_zero = da.line_cut(
    'zero',
    param=(0.0, 'x'),     # V fixed at 0.0; traverse U across full extent
    n_samples=256,
    width_px=1,
    show=True             # draws overlay + profile
)


**Line cut — on a captured image.** `'infinite'` is the perpendicular case: hold U, traverse V.

In [ ]:
# 2) Vertical line cut at fixed U ("infinite" preset) using the returned image directly
lc_inf = da.line_cut(
    'infinite',
    param=(0.0, 'y'),     # U fixed at 0.0; traverse V across full extent
    vol=(img, extent),
    n_samples=512,
    width_px=3,
    show=True
)


**Line cut — diagonals.** Corner to corner. Returns a dict; `lc['distance']` pairs with the intensities.

In [ ]:
# 3) Diagonals across the full slice extent
lc_pos = da.line_cut(
    'positive',
    n_samples=512,
    width_px=3,
    show=True
)

lc_neg = da.line_cut(
    'negative',
    n_samples=512,
    width_px=3,
    show=True
)

print('Positive cut samples:', len(lc_pos['distance']))


**Line cut — endpoints.** `((u0,v0), (u1,v1))` in slice coordinates, not pixels.

In [ ]:
# 4) Custom endpoints in U/V coordinates
lc_custom = da.line_cut(
    ((-0.25, -0.25), (0.25, 0.40)),  # endpoints in physical slice coordinates
    vol=(img, extent),
    n_samples=512,
    width_px=2,
    show=True
)


**Line cut — interactive.** Draggable endpoints, live profile. Needs `%matplotlib widget`.

In [ ]:
# 5) Interactive line cut with draggable endpoints (requires interactive matplotlib backend)
#    Ensure the first cell used '%matplotlib widget' and that 'ipympl' is installed.
#    You can drag the cyan endpoints and the magenta profile updates in real-time.
lc_interactive = da.line_cut(
    'positive',
    vol=(img, extent),
    interactive=True,
    n_samples=256,
    width_px=3
)


## Volumes

Convert the point cloud to a structured volume and visualize it, then slice that volume.

**Volume.** `create_vol` interpolates the points onto a regular grid — cheap to re-slice and render.

In [ ]:
# Create a 3D volume from points
vol = da.create_vol(data.points, data.intensities)
print("Created volume dimensions:", vol.dimensions)
print("Volume spacing:", vol.spacing)
print("Volume origin:", vol.origin)

# Visualize the volume
# Tip: try different colormaps like 'plasma', 'viridis', 'jet'
da.show_vol(vol, cmap='plasma')

# Slice the volume directly (canonical HL plane)
sl_from_vol = da.slice_data(
    data=vol,
    hkl='HL',
    shape=(400, 400),
    intensity_range=(None, None),   # no pre-filtering
    show=True,
    axis_display='hkl',
    show_grid=True
)


## Display options

Use min/max intensity thresholds and clim for display; switch label format; keep per-pixel physical size consistent when changing shape.

**Display.** Retune without re-slicing: `clim`, `min/max_intensity`, `shape_data=True` to keep aspect ratio.

In [ ]:
# Display with custom limits and HKL labels
_ = da.show_slice(
    sl_from_vol,
    cmap='coolwarm',
    clim=(0, 10000),
    min_intensity=100,          # filter out low intensity pixels
    max_intensity=50000,        # cap high intensity
    axis_display='hkl',
    show_grid=True,
    shape_data=True             # keep physical size per pixel when reshaping
)

# Display with U/V labels
_ = da.show_slice(
    sl_from_vol,
    cmap='viridis',
    axis_display='uv',
    show_grid=False
)


## Image-based workflow

Get the image/extent, perform line cuts without re-slicing, and reuse cached data for subsequent operations.

**Recap.** Slice → capture with `return_image=True` → cut the captured image.

In [ ]:
img2, extent2 = da.show_slice(sl_from_vol, return_image=True)
lc_from_image = da.line_cut(
    'zero',
    param=(0.0, 'x'),
    vol=(img2, extent2),
    n_samples=300,
    width_px=3,
    show=True
)


## Tips

- Interactive cuts need `%matplotlib widget` (`ipympl`).
- `hkl` takes a preset (`'HK'`, `'KL'`, `'HL'`) or a point plus `normal=(h,k,l)`.
- `axes=((h1,k1,l1), (h2,k2,l2)[, normal])` sets custom in-plane axes and labels them.
- `slab_thickness` widens a plane; pair with `intensity_range` to pre-filter.
- `return_image=True` gives `(img, extent)` to reuse without re-slicing.

## Next

- `RSM_Gridder.ipynb` — building gridded volumes, offline and live
- `DashPVA_Tools_Tour.ipynb` — file I/O, masking, Q conversion, settings